In [0]:
df_customer=spark.table("customer")
df_order=spark.table("orders")
df_invoice=spark.table("invoice")

In [0]:
# find 3rd highest purchase by customer
from pyspark.sql.functions import col,sum,row_number
from pyspark.sql.window import Window
df_customer.alias("c").join(df_order.alias("o"), col("c.cust_id") == col("o.cust_id"), "inner").groupBy("cust_name").agg(sum(col("order_amount")).alias("total")).orderBy(col("total").desc()).withColumn("rn",row_number().over(Window.orderBy(col("total").desc()))).select("cust_name","total").filter(col("rn")==3).show()

In [0]:
%sql
select cust_name,total from(select c.cust_name,sum(o.order_amount) as total,row_number() over(order by sum(o.order_amount) desc) as rn from customer c join orders o on c.cust_id=o.cust_id group by c.cust_name)t where rn=3

In [0]:


df_employee=spark.table("employee")
df_dept=spark.table("department")




In [0]:
%sql
select * from employee;
select * from department

In [0]:
# Get employee name with department name
from pyspark.sql.functions import *
df_employee.alias("e") \
.join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"inner") \
.select("e.emp_name","d.dept_name").show()

In [0]:
spark.sql("""select e.emp_name,d.dept_name from employee e join department d on e.dept_id=d.dept_id""").show()

In [0]:
# Get all employees without department
df_employee.alias("e") \
.join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"),"left")\
.select("e.emp_name","d.dept_name") \
.filter(col("d.dept_name").isNull()).show()

In [0]:
%sql
select e.emp_name,d.dept_name from employee e left join department d on e.dept_id=d.dept_id where d.dept_name is null

In [0]:
#Find total salary department-wise
from pyspark.sql.functions import *
df_employee.alias("e")\
    .join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"))\
        .groupBy("dept_name")\
            .agg(sum(col("salary")).alias("total_salary")).show()

In [0]:
%sql
select d.dept_name,sum(e.salary) as total_salary from employee e join department d on e.dept_id=d.dept_id group by d.dept_name

In [0]:
#Find average salary by department
from pyspark.sql.functions import *
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id")) \
.groupBy("dept_name") \
.agg(avg(col("salary")).alias("avg_salary"), \
     collect_list(col("d.dept_id")).alias("dept_ids")).select("dept_name","dept_ids","avg_salary").show()

In [0]:
%sql
select d.dept_name,collect_list(d.dept_id) as dept_ids,avg(e.salary) as avg_salary from employee e join department d on e.dept_id=d.dept_id group by d.dept_name

In [0]:
# Find highest salary in each department
from pyspark.sql.functions import *
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"))\
    .groupBy("d.dept_name").agg(max(col("salary")).alias("highest salary"))\
        .select("dept_name","highest salary").show()

In [0]:
%sql
select d.dept_name,max(e.salary) from employee e join department d on e.dept_id=d.dept_id group by d.dept_name

In [0]:
#Rank employees by salary within department
from pyspark.sql.window import Window
from pyspark.sql.functions import *
df_employee.alias("e")\
    .join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"))\
        .withColumn("rank",dense_rank().over(Window.partitionBy("dept_name")\
            .orderBy(col("salary").desc())))\
                .select("emp_name","dept_name","salary","rank").show()

In [0]:
%sql
select e.emp_name,d.dept_name,e.salary,dense_rank() over(partition by d.dept_name order by e.salary desc) as rank from employee e join department d on e.dept_id=d.dept_id

In [0]:
#Find highest paid employee in each department
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id")).withColumn("rank",dense_rank().over(Window.partitionBy("dept_name").orderBy(col("salary").desc())))\
    .filter(col("rank")==1)\
        .select("emp_name","dept_name","salary","rank").show()

In [0]:
%sql
select emp_name,dept_name,salary,rank from (select e.emp_name,d.dept_name,e.salary,dense_rank() over(partition by d.dept_name order by e.salary desc) as rank from employee e join department d on e.dept_id=d.dept_id)t
where rank=1

In [0]:
#Running total of salaries department-wise
df_employee.alias("e")\
    .join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id"))\
        .withColumn("running total",sum(col("salary"))\
            .over(Window.partitionBy("dept_name")\
                .orderBy(col("salary"))))\
                    .select("dept_name","salary","running total").show()

In [0]:
%sql
select d.dept_name,sum(e.salary) over(partition by d.dept_name order by e.salary) as running_total from employee e join department d on e.dept_id=d.dept_id

In [0]:
%sql
-- Get employee and manager names
select e.emp_name,m.emp_name as manager from employee e left join employee m on e.manager_id=m.emp_id


In [0]:
%sql
--Find employees earning more than their managers
SELECT e.emp_name AS employee,
       e.salary AS emp_salary,
       m.emp_name AS manager,
       m.salary AS mgr_salary
FROM employee e
JOIN employee m
ON e.manager_id = m.emp_id
WHERE e.salary > m.salary;


In [0]:
#find 2nd highest salary
from pyspark.sql.window import Window
from pyspark.sql.functions import col,dense_rank
df_employee.alias("e").join(df_dept.alias("d"),col("e.dept_id")==col("d.dept_id")).withColumn("dense_rank",dense_rank().over(Window.orderBy(col("salary").desc()))).filter(col("dense_rank")==3).select("emp_name","dept_name","salary").show()

In [0]:
%sql
select emp_name,max(salary) as salary from employee where salary <
(select max(salary) from employee where salary<
(select max(salary) from employee))
 group by emp_name order by salary desc limit(1)

In [0]:
#Find employees with salary above company average
from pyspark.sql.functions import avg,col
avg_salary=df_employee.select(avg(col("salary")).alias("avg_salary")).collect()[0][0]

df_employee.select("emp_name","salary").filter(col("salary")>avg_salary).show()


In [0]:
%sql
select emp_name,salary from employee where salary>(select avg(salary) as avg_salary from employee)

In [0]:
%sql
SELECT dept_id, COUNT(emp_id)
FROM employee
GROUP BY dept_id;